<a href="https://colab.research.google.com/github/haeshal25/myreposit/blob/main/ImageCaptioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip Flicker8k_Dataset.zip

Streaming output truncated to the last 5000 lines.
 extracting: 1650280501_29810b46e5.jpg  
 extracting: 2052953131_30834196fb.jpg  
 extracting: 2083778090_3aecaa11cc.jpg  
 extracting: 2089555297_95cf001fa7.jpg  
 extracting: 2195887578_3ba2f29b48.jpg  
 extracting: 2230067846_74046b89d3.jpg  
 extracting: 2264316030_600e55748d.jpg  
 extracting: 2271671533_7538ccd556.jpg  
 extracting: 2420730259_86e7f8a815.jpg  
 extracting: 2602085456_d1beebcb29.jpg  
 extracting: 2687672606_275169c35d.jpg  
 extracting: 2750867389_4b815f793a.jpg  
 extracting: 2751694538_fffa3d307d.jpg  
 extracting: 2809793070_1a3387cd6e.jpg  
 extracting: 2871962580_b85ce502ba.jpg  
 extracting: 2929272606_2a5923b38e.jpg  
 extracting: 3015898903_70bebb8903.jpg  
 extracting: 3018847610_0bf4d7e43d.jpg  
 extracting: 3172384527_b107385a20.jpg  
 extracting: 3246804978_ea2c9e56f2.jpg  
 extracting: 3268191118_ba25fabab6.jpg  
 extracting: 3288173388_03bc2a844d.jpg  
 extracting: 3451523035_b61d79f6a8.jpg  
 extra

In [ ]:
!wget  https://raw.githubusercontent.com/text-machine-lab/MUTT/refs/heads/master/data/flickr/Flickr8k.token.txt

--2025-03-26 05:20:00--  https://raw.githubusercontent.com/text-machine-lab/MUTT/refs/heads/master/data/flickr/Flickr8k.token.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3395237 (3.2M) [text/plain]
Saving to: ‘Flickr8k.token.txt’

Flickr8k.token.txt  100%[===================>]   3.24M  --.-KB/s    in 0.08s   

2025-03-26 05:20:00 (41.8 MB/s) - ‘Flickr8k.token.txt’ saved [3395237/3395237]



In [ ]:
!wget -q --show-progress http://nlp.stanford.edu/data/glove.42B.300d.zip

glove.42B.300d.zip  100%[===================>]   1.75G   182KB/s    in 59m 24s 


In [ ]:
!unzip -q glove.42B.300d.zip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, add, Dropout,\
Activation,Flatten, BatchNormalization, RepeatVector, TimeDistributed,Reshape,concatenate

from tensorflow.keras.models import Model, load_model
import tensorflow.keras.preprocessing.image as tf_image
import tensorflow.keras.applications.inception_v3 as inception

import string
from string import punctuation
from nltk import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import re
from tqdm import tqdm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
encode_model = InceptionV3(weights='imagenet')
encode_model = Model(encode_model.input, encode_model.layers[-2].output)
WIDTH = 299
HEIGHT = 299
OUTPUT_DIM = 2048
START = "startseq"
STOP = "endseq"
EPOCHS = 10
preprocess_input = inception.preprocess_input

96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
import tensorflow.keras.preprocessing.image as tf_image

In [ ]:
def encodeImage(img):
  img = img.resize((WIDTH, HEIGHT))

  x = tf_image.img_to_array(img)

  x = np.expand_dims(x, axis=0)

  x = preprocess_input(x)

  x = encode_model.predict(x) # Get the encoding vector for the image
  x = np.reshape(x, OUTPUT_DIM )

  return x

In [ ]:
import pandas as pd
token = pd.read_csv('Flickr8k.token.txt', delimiter='\t', header=None, names=['img','caption'])
token.head()

,img,caption
0,1000268201_693b08cb0e.jpg#0,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg#1,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg#2,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg#3,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg#4,A little girl in a pink dress going into a woo...


In [ ]:
token['img_id'] = token['img'].apply(lambda x: x.split('.')[0])
token.head(2)

,img,caption,img_id
0,1000268201_693b08cb0e.jpg#0,A child in a pink dress is climbing up a set o...,1000268201_693b08cb0e
1,1000268201_693b08cb0e.jpg#1,A girl going into a wooden building .,1000268201_693b08cb0e


In [ ]:
token['img'] = token['img'].apply(lambda x: x.split('#')[0])
token['img'].head(2)

,img
0,1000268201_693b08cb0e.jpg
1,1000268201_693b08cb0e.jpg


In [ ]:
encode_model = InceptionV3(weights='imagenet')
encode_model = Model(encode_model.input, encode_model.layers[-2].output)
# we r not interested in classification we want the 2nd last layer
WIDTH = 299
HEIGHT = 299
OUTPUT_DIM = 2048
START = "startseq"  # start tag
STOP = "endseq"     # end tag to differentiate the usual caption
EPOCHS = 10
preprocess_input = inception.preprocess_input

In [ ]:
token['caption'][0:2]

,caption
0,A child in a pink dress is climbing up a set o...
1,A girl going into a wooden building .


In [ ]:
token['caption'] = token['caption'].apply(lambda x: START+' '+x+' '+STOP)
token['caption'][1]

'startseq A girl going into a wooden building . endseq'

In [ ]:
token['caption'] = token['caption'].apply(lambda x: re.sub('['+punctuation+']',' ',x))
token['caption'] = token['caption'].apply(lambda x: re.sub("\d"," ", x))
token['caption'] = token['caption'].replace('-', ' ')
token['caption'] = token['caption'].apply(lambda x: re.sub("\s+"," ", x))
token['caption'] = token['caption'].str.lower()

In [ ]:
word_count_threshold = 5
word_counts ={}
for caption in token['caption']:
  for w in word_tokenize(caption):
    word_counts[w] = word_counts.get(w,0) + 1

In [ ]:
vocab = [w for w in word_counts if word_counts[w] >= word_count_threshold]
print('preprocessed word count %d ==> %d' % (len(word_counts), len(vocab)))

preprocessed word count 8442 ==> 2973


In [ ]:
caption_lens=[]
for caption in token['caption']:
  words=word_tokenize(caption)
  words=[w for w in words if w in vocab]
  caption_lens.append(len(words))
max_length=max(caption_lens)

In [ ]:
print('maximum caption length is:',max_length)

maximum caption length is: 37


In [ ]:
idxtoword = {}
wordtoidx = {}

ix = 1
for w in vocab:
    wordtoidx[w] = ix
    idxtoword[ix] = w
    ix += 1

vocab_size = len(idxtoword) + 1
vocab_size

2974

In [ ]:
embeddings_index = {}
f = open( 'glove.42B.300d.txt', encoding="utf-8")

for line in f:
    line=line.strip()
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs

f.close()
print(f'Found {len(embeddings_index)} word vectors.')

Found 1917494 word vectors.


In [ ]:
embedding_dim = 300

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in wordtoidx.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [ ]:
embedding_matrix.shape

(2974, 300)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

In [ ]:
def data_generator(data, encoded_images, wordtoidx, max_length, num_photos_per_batch):
  # x1 - Training data for photos
  # x2 - The caption that goes with each photo
  # y - The predicted rest of the caption
  x1, x2, y = [], [], []
  n=0
  while True:
    for k,caption in enumerate(data['caption']):
      n+=1
      photo = encoded_images[data['id'][k]]

      seq = [wordtoidx[word] for word in word_tokenize(caption) if word in wordtoidx]
        # Generate a training case for every possible sequence and outcome
      for i in range(1, len(seq)):
        in_seq, out_seq = seq[:i], seq[i]
        in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
        out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
        x1.append(photo)
        x2.append(in_seq)
        y.append(out_seq)
      if n==num_photos_per_batch:
        yield ([np.array(x1), np.array(x2)], np.array(y))
        # next when generator gets called iteration will start from where we left off
        # this makes it make a pass through the complete data in an epoch
        x1, x2, y = [], [], []
        n=0

In [ ]:
from tensorflow.keras.layers import LSTM, Embedding, TimeDistributed, Dense, RepeatVector,\
                         Activation, Flatten, Reshape, concatenate, Dropout, BatchNormalization,add

In [ ]:
inputs1 = Input(shape=(OUTPUT_DIM,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)
caption_model = Model(inputs=[inputs1, inputs2], outputs=outputs)

In [ ]:
caption_model.layers[2].set_weights([embedding_matrix])
caption_model.layers[2].trainable = False
caption_model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
from tqdm import tqdm

In [ ]:
caption_model.optimizer.lr = 1e-4
number_pics_per_batch = 6
steps = len(data['caption'])//number_pics_per_batch

for i in tqdm(range(EPOCHS)):
    generator = data_generator(data, encoded_images, wordtoidx, max_length, number_pics_per_batch)
    caption_model.fit_generator(generator, epochs=1, steps_per_epoch=steps, verbose=1)

In [ ]:
def generateCaption(photo):
    in_text = START
    for i in range(max_length):
        sequence = [wordtoidx[w] for w in in_text.split() if w in wordtoidx]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = caption_model.predict([photo,sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = idxtoword[yhat]
        in_text += ' ' + word
        if word == STOP:
            break
    final = in_text.split()
    final = final[1:-1]
    final = ' '.join(final)
    return final

In [ ]:



image=encoded_images[int(image_file.split('.')[0])]

image = image.reshape((1,OUTPUT_DIM))
x=plt.imread('Flicker8k.token.text'+image_file)
plt.imshow(x)
plt.show()
print("Caption:",generateCaption(image))